In [2]:
import duckdb, pandas as pd

con = duckdb.connect("../data/processed/recalls.duckdb")

EXCLUDE = ["VEHICLE SPEED CONTROL", "SUSPENSION", "SEATS"]
pred = pd.read_parquet("../data/processed/unknown_predictions.parquet")
rec = pred[(pred["confidence"] >= 0.9) & (~pred["pred_system"].isin(EXCLUDE))].copy()
rec["ODINO"] = rec["ODINO"].astype(str)
print(f"{len(rec):,} recovered complaints")

con.register("rec_df", rec)
con.execute("CREATE OR REPLACE TABLE recovered AS SELECT ODINO, pred_system FROM rec_df")
con.execute("""
CREATE OR REPLACE TABLE complaints_aug AS
SELECT c.* REPLACE (COALESCE(r.pred_system, c.system_norm) AS system_norm)
FROM complaints c
LEFT JOIN recovered r
  ON c.ODINO = r.ODINO AND c.system_norm = 'UNKNOWN OR OTHER'
""")

print(con.execute("""
SELECT
  (SELECT COUNT(DISTINCT ODINO) FROM complaints
    WHERE system_norm <> 'UNKNOWN OR OTHER') AS known_before,
  (SELECT COUNT(DISTINCT ODINO) FROM complaints_aug
    WHERE system_norm <> 'UNKNOWN OR OTHER') AS known_after
""").df().to_string(index=False))

20,666 recovered complaints
 known_before  known_after
       731730       752396


In [3]:
sql = open("../sql/02_panel.sql").read()
sql_aug = (sql.replace("FROM complaints", "FROM complaints_aug")
              .replace("CREATE OR REPLACE TABLE panel AS",
                       "CREATE OR REPLACE TABLE panel_aug AS"))

for stmt in [s for s in sql_aug.split(";") if s.strip()]:
    con.execute(stmt)

print(con.execute("""
SELECT 'original' AS panel, COUNT(*) AS rows, SUM(target) AS positives,
       COUNT(DISTINCT campaign) FILTER (WHERE target = 1) AS recalls
FROM panel
UNION ALL
SELECT 'augmented', COUNT(*), SUM(target),
       COUNT(DISTINCT campaign) FILTER (WHERE target = 1)
FROM panel_aug
""").df().to_string(index=False))
con.close()

    panel    rows  positives  recalls
 original 2363007    20681.0     1667
augmented 2393219    20902.0     1678


In [4]:
import numpy as np, lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

FEATURES = ["make", "system", "n_1m", "n_3m", "n_6m", "n_12m", "n_prev3m",
            "crash_12m", "fire_12m", "injury_12m", "towed_12m",
            "vehicle_age", "months_since_first", "vehicle_12m", "system_share",
            "sibling_years_12m", "family_recalled"]
CAT = ["make", "system"]
K = 100

def evaluate(table):
    c = duckdb.connect("../data/processed/recalls.duckdb", read_only=True)
    df = c.execute(f"SELECT * FROM {table}").df()
    c.close()
    df["month"] = pd.to_datetime(df["month"])
    for col in CAT:
        df[col] = df[col].astype("category")

    tr = df[df["month"] < "2022-01-01"]
    te = df[df["month"] >= "2023-01-01"].copy()

    m = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.05, num_leaves=63,
                           min_child_samples=200, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(tr[FEATURES], tr["target"], categorical_feature=CAT)
    te["p"] = m.predict_proba(te[FEATURES])[:, 1]
    te["rank"] = te.groupby("month")["p"].rank(ascending=False, method="first")

    top = te[te["rank"] <= K]
    represented = te.loc[te["target"] == 1, ["campaign", "initiated_by"]].drop_duplicates()
    caught = top.loc[top["target"] == 1, ["campaign", "initiated_by"]].drop_duplicates()

    out = {"panel": table,
           "ROC-AUC": roc_auc_score(te["target"], te["p"]),
           "PR-AUC": average_precision_score(te["target"], te["p"]),
           f"precision@{K}": top["target"].mean(),
           "recalls_represented": represented["campaign"].nunique(),
           f"caught@{K}": caught["campaign"].nunique()}
    for g in ["MFR", "ODI"]:
        out[f"{g}_represented"] = int((represented["initiated_by"] == g).sum())
        out[f"{g}_caught"] = int((caught["initiated_by"] == g).sum())
    return out

res = pd.DataFrame([evaluate("panel"), evaluate("panel_aug")]).set_index("panel").T
print(res.round(4).to_string())
res.to_csv("../results/augmentation_comparison.csv")

panel                   panel  panel_aug
ROC-AUC                0.8105     0.8099
PR-AUC                 0.0383     0.0390
precision@100          0.0735     0.0790
recalls_represented  648.0000   654.0000
caught@100            76.0000    80.0000
MFR_represented      507.0000   512.0000
MFR_caught            53.0000    62.0000
ODI_represented      132.0000   133.0000
ODI_caught            23.0000    18.0000
